# First order Runge-Kutta methods

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differential_equations/initial_value_problems/explicit_methods/first_order_runge_kutta.ipynb)

In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Forward Euler Method

The Forward Euler method (also known as the Euler-Cauchy / point-slope method, or Explicit Euler) is the simplest Runge-Kutta method. One simply takes the slope at the current point, given by the differential equation, and assumes it remains constant over the step:

Given

$$
\frac{\mathrm{d}y}{\mathrm{d}t} = f(t, y)
$$

the next time step is:

$$
y_{i+1} = y_i + f(t_i, y_i) h
$$

which is equivalent to the left Riemann sum, carrying $O(h^2)$ local error over the step and $O(h)$ global error over the full solution.

```{literalinclude} ../../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:linenos:
:start-at: def forward_euler
:end-before: def heun_method
```

In [ ]:
exe.display_source(exe.forward_euler)

def f(t, y):
    return 4.0 * np.exp(0.8 * t) - 0.5 * y

def y_true(t):
    return (4.0 / 1.3) * (np.exp(0.8 * t) - np.exp(-0.5 * t)) + 2.0 * np.exp(-0.5 * t)

res_euler = exe.forward_euler(f, t0=0.0, y0=2.0, t_end=4.0, h=1.0)
fig = exe.plot_ode_comparison((0, 4), y_true, {"Forward Euler (h=1)": res_euler})
fig.show()

## Heun's Method

Now that we have a *prediction* for $y_{i+1}$, can we use this as an initial guess and *correct* our slope estimate? Heun's method does exactly this and is called a **predictor-corrector** algorithm.

Consider $\frac{\mathrm{d}y}{\mathrm{d}t} = f(t, y)$ and make a Forward Euler prediction:

$$
y^0_{i+1} = y_i + f(t_i, y_i) h
$$

Now take the average of the slopes at the start and end of the interval:

$$
\begin{aligned}
\bar{y}' &= \frac{y'_i + y'_{i+1}}{2} \\
&= \frac{f(t_i, y_i) + f(t_{i+1}, y^0_{i+1})}{2}
\end{aligned}
$$

which is used in the corrector:

$$
y_{i+1} = y_i + \frac{f(t_i, y_i) + f(t_{i+1}, y^0_{i+1})}{2} h
$$

```{literalinclude} ../../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:linenos:
:start-at: def heun_method
:end-before: def midpoint_ode
```

In [ ]:
exe.display_source(exe.heun_method)

### Example: Heun's Method with Multiple Predictor-Corrector Cycles

One could repeat the predictor-corrector cycle iteratively with the intention of converging toward the exact answer, but we will see why this does not eliminate truncation error.

Integrate

$$
y' = 4 e^{0.8 t} - 0.5 y, \quad y(0) = 2
$$

using Heun's method with $1$ and $15$ predictor-corrector cycles ($h = 1$). Unfortunately, iterating Heun's corrector converges to the fixed point of the trapezoidal discretization, not the exact analytical solution!

In [ ]:
res_heun_1 = exe.heun_method(f, t0=0.0, y0=2.0, t_end=4.0, h=1.0, iterations=1)
res_heun_15 = exe.heun_method(f, t0=0.0, y0=2.0, t_end=4.0, h=1.0, iterations=15)

for t_val, y1, y15 in zip(res_heun_1["t_points"][1:], res_heun_1["y_points"][1:], res_heun_15["y_points"][1:]):
    print(f"t = {t_val:.0f} | Heun (1 iter) = {y1:.4f} | Heun (15 iter) = {y15:.4f} | Exact = {y_true(t_val):.4f}")

fig = exe.plot_heun_iterations(f, y_true, t0=0.0, y0=2.0, t_end=4.0, h=1.0, max_iter=15)
fig.show()

### Special case: connection to the Trapezoid Rule

In the special case where the slope does not depend on the dependent variable, $y'(t) = f(t)$, the endpoint slope is known directly without a predictor cycle:

$$
y_{i+1} = y_i + \frac{f(t_i) + f(t_{i+1})}{2} h
$$

which is simply the composite trapezoid rule, carrying $O(h^3)$ local accuracy and $O(h^2)$ global accuracy.

## The Midpoint Method

Recall from differentiation and integration that sampling at the midpoint is often superior to either endpoint in isolation because leading over- and under-estimates cancel.

As an alternative to Heun's predictor-corrector method, we subdivide the interval and estimate the state at the *midpoint* via a half-step:

$$
y_{i+1/2} = y_i + f(t_i, y_i) \frac{h}{2}
$$

We then evaluate the slope at the midpoint, $y'_{i+1/2} = f(t_{i+1/2}, y_{i+1/2})$, to advance the full step:

$$
y_{i+1} = y_i + f(t_{i+1/2}, y_{i+1/2}) h
$$

which achieves $O(h^3)$ local and $O(h^2)$ global error. The midpoint method demonstrates that we can calculate and combine *substeps* to achieve higher-order predictions. 

**Optimal construction of substeps will be the foundation for the general Runge-Kutta methods.**

```{literalinclude} ../../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:linenos:
:start-at: def midpoint_ode
:end-before: def ralston_method
```

In [ ]:
exe.display_source(exe.midpoint_ode)

res_mid = exe.midpoint_ode(f, t0=0.0, y0=2.0, t_end=4.0, h=1.0)
fig = exe.plot_ode_comparison(
    (0, 4),
    y_true,
    {"Forward Euler": res_euler, "Heun's Method": res_heun_1, "Midpoint Method": res_mid},
)
fig.show()